# HP-4 · LoRA cho dịch máy đẳng thời (NLLB-200)

**Ý tưởng.** Dịch máy thường chỉ tối ưu *đúng nghĩa*. Lồng tiếng còn cần *đúng độ
dài*: câu tiếng Việt phải đọc vừa khung thời gian của câu gốc. Ở đây ta dạy
NLLB-200 nhận thêm một **token ngân sách** ở đầu câu nguồn — `<len:12>` nghĩa là
"hãy dịch sao cho đọc khoảng 12 âm tiết" — rồi fine-tune bằng LoRA.

**Vì sao LoRA.** NLLB-200 distilled 600M có ~600 triệu tham số. LoRA chỉ huấn
luyện vài triệu tham số thêm vào, vừa một GPU phổ thông, và giữ nguyên mô hình gốc.

**Trạng thái: ĐÃ CHẠY** — Trung → Việt (11/11/2026); Nhật, Hàn, Anh → Việt (20–21/11/2026),
**mỗi ngôn ngữ nguồn một adapter**. Huấn luyện thật không chạy trong notebook mà bằng script
dòng lệnh, vì cần chạy tiếp được từ checkpoint và một môi trường riêng có torch CUDA:

- `training/prepare_opus.py` — lọc OPUS OpenSubtitles từng thứ tiếng (vi–zh_CN: 13,5 triệu cặp → 7,9 triệu đạt chuẩn → lấy mẫu 300.000; Nhật, Hàn, Anh cũng 300.000)
- `training/build_corpus.py --extra … --holdout …` — gộp phần Trung với dữ liệu của tool, loại tập kiểm thử
- `training/train_nllb_lora.py --src zh|ja|ko|en` — huấn luyện (RTX 4060 Laptop 8 GB: 1 epoch, ~9.350 bước, 42–93 phút mỗi adapter)
- `training/eval_nllb_lora.py` — đo trên bộ gold và bảng chéo 7 thứ tiếng; `training/bootstrap_cross.py` — khoảng tin cậy 95 %

Notebook này giữ lại để **giải thích ý tưởng và dịch thử**; bảng số đo ở cuối và trong
`training/README.md`. Các ô huấn luyện vẫn có khoá `CHO_PHEP_CHAY`.

**Nhóm A (huấn luyện).** Trung, Nhật, Hàn, Anh → Việt — **đã làm đủ bốn.**
**Nhóm B (chỉ kiểm tra zero-shot).** Pháp, Nga, Thái → Việt: mỗi thứ tiếng 500 câu kiểm tra, không
adapter nào học. Đây là giả thuyết H1 của đề cương: ràng buộc độ dài học được có lan sang ngôn ngữ
chưa huấn luyện không. **Đã đo: có lan sang**, theo phép đo trên văn bản (bảng ở cuối notebook).

In [ ]:
# Chạy được dù mở notebook từ thư mục nào: đi ngược lên tới thư mục có backend/
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "backend").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Gốc dự án:", ROOT)

## Kiểm tra môi trường

In [ ]:
import importlib.util

need = {
    "torch": "--extra offline (hoặc --extra clone)",
    "transformers": "--extra offline",
    "peft": "uv pip install peft",
    "datasets": "uv pip install datasets",
}
missing = [n for n in need if importlib.util.find_spec(n) is None]

for name, how in need.items():
    mark = "✓" if name not in missing else "✗"
    print(f"  {mark} {name:<14} {'' if name not in missing else how}")

if missing:
    print("\nCòn thiếu:", ", ".join(missing))
else:
    import torch
    print("\nGPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ — LoRA sẽ rất chậm")

## Chuẩn bị dữ liệu

Mỗi mẫu gồm câu nguồn, câu đích tiếng Việt, và **ngân sách âm tiết** tính từ
chính câu đích trong tập huấn luyện. Lúc dùng thật thì ngân sách lấy từ khung
thời gian của cue chia cho tốc độ đọc (xem `backend/core/budget.py`).

In [ ]:
from backend.core import duration, isochron

# Cách gắn token nằm trong backend/core/isochron.py để script huấn luyện và app dùng chung
to_example = isochron.to_example

demo = [
    to_example("我們獲勝了", "Chúng ta đã thắng rồi"),
    to_example("I am the Cyclops and I will eat you all", "Ta là gã khổng lồ một mắt, ta sẽ ăn thịt các ngươi"),
]
for d in demo:
    print(d["src"])
    print("  ->", d["tgt"], f'({duration.syllables(d["tgt"])} âm tiết)')

In [ ]:
# Nạp corpus song ngữ thật. Định dạng mong đợi: JSONL mỗi dòng {"src": ..., "tgt": ...}
# HP-6 của đề cương sinh ra file này từ phụ đề OpenSubtitles.
CORPUS = ROOT / "data" / "corpus" / "vidub_train.jsonl"

if CORPUS.exists():
    import json

    pairs = [json.loads(ln) for ln in CORPUS.read_text(encoding="utf-8").splitlines() if ln.strip()]
    examples = [to_example(p["src"], p["tgt"]) for p in pairs]
    print(f"{len(examples)} mẫu từ {CORPUS}")
else:
    examples = demo
    print(f"Chưa có {CORPUS} — dùng tạm {len(demo)} mẫu minh hoạ.")
    print("HP-6 (dựng bộ dữ liệu ViDub) phải xong trước bước này.")

## Cấu hình LoRA

In [ ]:
from backend.config import MODEL_DIR

BASE_MODEL = "facebook/nllb-200-distilled-600M"

LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"], task_type="SEQ_2_SEQ_LM")
# Cấu hình đã chạy thật trên GPU 8 GB: batch 16 × tích luỹ 2, mô hình gốc để half precision
TRAIN = dict(learning_rate=2e-4, num_train_epochs=1,
             per_device_train_batch_size=16, gradient_accumulation_steps=2)

print("Mô hình gốc:", BASE_MODEL)
print("LoRA        :", LORA)
print("Huấn luyện  :", TRAIN)
# mỗi ngôn ngữ nguồn một thư mục: nllb_lora_dangthoi (Trung), …_ja, …_ko, …_en — cùng 60 token ngân sách
for lang in isochron.LANGS:
    print(f"Adapter {lang}:", isochron.adapter_dir(MODEL_DIR, lang),
          "(đã có)" if isochron.has_adapter(MODEL_DIR, lang) else "(chưa huấn luyện)")

## Huấn luyện

Chạy thật bằng dòng lệnh (dừng êm sau `--max-minutes`, chạy lại là tiếp tục từ checkpoint):

```bash
data/_train_venv/Scripts/python training/train_nllb_lora.py --origins extra --dev 500   # Trung → Việt
data/_train_venv/Scripts/python training/train_nllb_lora.py --src ko --dev 500          # cũng: --src ja, --src en
```

Ô dưới đây là bản rút gọn để đọc hiểu, gọi đúng các hàm của script. Hai điều notebook
bản đầu làm thiếu và chạy thật mới lộ ra:

1. `add_tokens` tạo 60 hàng embedding **mới, ngẫu nhiên**; LoRA trên `q_proj/v_proj` không
   đụng tới embedding nên phải cho huấn luyện riêng 60 hàng đó (`trainable_token_indices`).
2. Batch 32 ở fp32 chiếm hơn 8 GB → Windows tràn sang RAM dùng chung, tốc độ rơi còn
   một nửa. Để mô hình gốc ở half precision, batch 16 × tích luỹ 2 thì vừa (≈ 92 mẫu/giây).

⚠ Đổi `CHO_PHEP_CHAY = True` sẽ huấn luyện lại ~1 giờ trên GPU.

In [ ]:
CHO_PHEP_CHAY = False   # đổi thành True để huấn luyện lại (~1 giờ trên RTX 4060)
NGON_NGU = "zh"         # ngôn ngữ nguồn của adapter: zh, ja, ko hoặc en

if not CHO_PHEP_CHAY:
    print("Đang khoá. Đặt CHO_PHEP_CHAY = True để chạy thật.")
elif missing:
    print("Thiếu thư viện:", ", ".join(missing))
else:
    import subprocess

    # chạy đúng script dòng lệnh, để notebook và bản chạy thật không lệch nhau
    cmd = [sys.executable, str(ROOT / "training" / "train_nllb_lora.py"), "--src", NGON_NGU, "--dev", "500"]
    if NGON_NGU == "zh":
        cmd += ["--origins", "extra"]  # phần Trung: chỉ dữ liệu OPUS đã gộp, không lẫn bản người dùng sửa
    subprocess.run(cmd, check=True)

## Dịch thử với ngân sách khác nhau

Cùng một câu, ép ngân sách ngắn dần — câu dịch phải ngắn theo. Ô thứ hai thử cùng một ý ở bốn
thứ tiếng nguồn: app tự bật adapter của đúng ngôn ngữ nguồn (`translate.lora_for`).

In [ ]:
def dich_voi_ngan_sach(text_nguon: str, n_am_tiet: int | None, nguon: str = "zh") -> str:
    """Dịch một câu sang tiếng Việt, ép độ dài về khoảng `n_am_tiet` âm tiết (None = dịch tự do)."""
    from backend.core import translate

    if not translate.lora_applies(nguon, "vi") or not translate.lora_available():
        return f"(chưa có adapter {nguon} → vi, hoặc thiếu thư viện peft)"
    # đúng hàm app gọi: nạp NLLB gốc + mọi adapter, bật adapter của ngôn ngữ nguồn
    return translate._translate_nllb([text_nguon], nguon, "vi", budgets=[n_am_tiet])[0]


for n in (None, 6, 10, 16):
    print(f"<len:{n}> ->", dich_voi_ngan_sach("我是独眼巨人，我要把你们都吃了", n))

In [ ]:
# Cùng một ý ở bốn thứ tiếng nguồn đã có adapter
CAU = {
    "zh": "天亮之前我们必须离开这里",
    "ja": "夜が明ける前にここを出なければならない",
    "ko": "날이 밝기 전에 여기를 떠나야 해",
    "en": "We have to get out of here before dawn",
}
for nguon, text in CAU.items():
    for n in (None, 5, 12):
        print(f"{nguon} <len:{n}> ->", dich_voi_ngan_sach(text, n, nguon))

## Đánh giá

Hai chỉ số đi cùng nhau, vì siết độ dài mà hỏng nghĩa thì vô nghĩa:

1. **BLEU / chrF** — nghĩa có giữ được không (`data/_bench/score_vi.py`).
2. **Tỉ lệ câu vượt khung** — đo bằng chính `budget.assess()` mà bản chạy thật dùng.

```bash
data/_train_venv/Scripts/python training/eval_nllb_lora.py
```

**Bộ gold** — 271 câu thoại phim cổ trang, dịch từng dòng, 68 đơn vị đọc:

| Bản dịch | BLEU | chrF | Vượt khung (đơn vị đọc) | Vượt khung (dòng) |
|---|---|---|---|---|
| NLLB gốc | 5,0 | 22,1 | 6/68 = 8,8 % | 67/271 |
| NLLB gốc + luật rút gọn (HP-5) | 5,0 | 22,1 | 5/68 = 7,4 % | 66/271 |
| NLLB + LoRA, không token | 5,6 | 22,3 | 6/68 = 8,8 % | 71/271 |
| NLLB + LoRA, dịch lại từng dòng với `<len:n>` | 5,9 | 22,0 | **0/68** | 56/271 |
| NLLB + LoRA, bộ siết của app (dịch lại cả đơn vị) | 5,5 | 20,5 | **0/68** | 51/271 |
| *Google (mốc)* | *7,9* | *28,9* | | |
| *LLM có ngữ cảnh (mốc)* | *41,6* | *63,3* | | |

**Tập dev Trung → Việt** (11/11/2026) — 500 cặp OpenSubtitles không tham gia huấn luyện, n = số âm tiết của bản tham chiếu:

| Bản dịch | BLEU | chrF | Lệch trung bình (âm tiết) | Trong ±1 âm tiết |
|---|---|---|---|---|
| NLLB gốc | 12,0 | 30,2 | 2,06 | 50,4 % |
| NLLB + LoRA, không token | 13,3 | 30,9 | 1,90 | 55,2 % |
| NLLB + LoRA, `<len:n>` | 13,0 | 31,0 | **1,01** | **81,6 %** |
| NLLB + LoRA, `<len:0,7n>` (ép ngắn 30 %) | 12,1 | 28,8 | 1,23 | 73,4 % |

**Bảng chéo 7 thứ tiếng** (21/11/2026) — mọi adapter trên mọi tập kiểm tra, gồm Pháp, Nga, Thái mà
không adapter nào học (adapter Trung: bản Hán Việt 21/11, đang dùng). 451–500 câu mỗi tập, đã loại câu có bản Việt trùng dữ liệu huấn luyện;
n = số âm tiết bản tham chiếu; *phải nén* = dài hơn n quá 5 % (ngưỡng app dùng để tăng tốc giọng).
Nhóm A: adapter đúng thứ tiếng; nhóm B: khoảng thấp nhất – cao nhất qua bốn adapter (zero-shot).

| Tập kiểm tra | Trong ±1 âm tiết: gốc → `<len:n>` | Phải nén: gốc → `<len:n>` | chrF: gốc → `<len:n>` |
|---|---|---|---|
| Trung | 49,0 → **83,0 %** | 43,9 → 32,3 % | 30,1 → 31,1 |
| Nhật | 46,3 → **82,3 %** | 33,0 → 27,3 % | 25,6 → 26,9 |
| Hàn | 40,5 → **76,9 %** | 37,7 → 28,2 % | 26,1 → 27,6 |
| Anh | 52,0 → **73,2 %** | 50,1 → 36,3 % | 40,4 → 41,3 |
| Pháp (chưa học) | 43,0 → 57,8–64,2 % | 37,2 → 27,0–33,0 % | 32,7 → 32,7–33,4 |
| Nga (chưa học) | 50,2 → 62,8–73,6 % | 44,8 → 25,8–37,6 % | 32,0 → 31,6–32,3 |
| Thái (chưa học) | 47,8 → 55,8–74,2 % | 39,6 → 23,0–37,4 % | 31,8 → 32,6–32,9 |

Bảng đủ từng ô (4 adapter × 7 tập) và khoảng tin cậy: `training/README.md`.

**Đọc bảng cho đúng.**

- Cái LoRA làm được là **điều khiển độ dài**: bám ngân sách ±1 âm tiết từ 40–52 % lên 73–83 % ở
  cả bốn thứ tiếng đã học, và trên bộ gold hết hẳn đơn vị đọc vượt khung (bộ luật HP-5 chỉ gỡ
  được 1/6).
- **Chất lượng dịch gần như không đổi.** BLEU bộ gold 5,0 → 5,6 chưa có ý nghĩa thống kê
  (bootstrap ghép cặp 1.000 lượt: LoRA thắng 74 % số lượt); bản Hán Việt 21/11 lên 6,7 — hơn NLLB gốc
  có ý nghĩa (mục cuối). Trên tập dev chrF tăng 0,8–1,4. NLLB dù có
  LoRA vẫn kém Google và kém xa LLM trên phim cổ trang: OpenSubtitles phần lớn là phim Âu–Mỹ,
  không dạy được "lang quân / thiếp / tại hạ".
- **Giả thuyết H1: có.** Dùng cho tiếng Pháp, Nga, Thái — chưa adapter nào học — token vẫn kéo độ dài
  về gần n (±1 âm tiết tăng 8–26 điểm, có ý nghĩa) mà chrF không giảm có ý nghĩa. Với adapter
  Trung, Nhật hoặc Hàn, tỉ lệ phải nén dưới 35 % trên cả ba thứ tiếng (tiêu chí MT-5). Khả năng
  bám độ dài gắn với phía tiếng Việt, không với ngôn ngữ nguồn đã học.
- **Phải nén vẫn 27–36 %** dù có token (MT-4 đặt dưới 25 %): token nhắm đúng n nên khoảng một phần ba
  bản dịch dài hơn n một hai âm tiết. Trong app, câu vượt khung được dịch lại với token nhỏ hơn.
- Ép ngắn 30 % thì mô hình chỉ ngắn đi được một phần (dài/n = 1,16) và mất ~2 điểm chrF —
  ngân sách là "mong muốn", không phải ràng buộc cứng.
- Bộ siết của app dịch lại cả đơn vị đọc rồi chia lại theo tỉ lệ ký tự nên điểm theo dòng
  thấp hơn (chrF 20,5) dù cũng hết vượt khung.
- Chưa làm: đo tỉ lệ câu phải nén trong một lần lồng tiếng thật; COMET (cần tải mô hình ~2,3 GB).

In [ ]:
import json

from backend.core import budget, metrics

print("BLEU:", "có sacrebleu" if metrics.bleu(["a"], ["a"]) is not None else "cần: uv pip install sacrebleu")
print("Ngưỡng vượt khung dùng chung:", budget.TOLERANCE)

ket_qua = isochron.adapter_dir(MODEL_DIR) / "eval_gold.json"
if ket_qua.exists():
    for r in json.loads(ket_qua.read_text(encoding="utf-8"))["gold"]:
        print(f"{r['name']:16s} BLEU {r['bleu']:5.1f}  chrF {r['chrf']:5.1f}  vượt khung {r['over_units']}/{r['units']}")
else:
    print("Chưa có số đo bộ gold — chạy training/eval_nllb_lora.py")

bang_cheo = ROOT / "data" / "_bench" / "nllb_lora_cross.json"
if bang_cheo.exists():
    rows = json.loads(bang_cheo.read_text(encoding="utf-8"))["rows"]
    get = {(r["lang"], r["run"]): r for r in rows}
    adapters = [a for a in isochron.LANGS if any(r["run"] == f"{a}+len" for r in rows)]
    for key, ten in (("within_1_pct", "trong ±1 âm tiết"), ("over_tol_pct", "phải nén")):
        print(f"\n{ten} (%) — dòng: tập kiểm tra; cột: NLLB gốc, rồi từng adapter + <len:n>")
        print(f"{'tập':5s} {'gốc':>6s} " + " ".join(f"{a:>6s}" for a in adapters))
        for lang in dict.fromkeys(r["lang"] for r in rows):
            print(f"{lang:5s} {get[(lang, 'goc')][key]:6.1f} "
                  + " ".join(f"{get[(lang, a + '+len')][key]:6.1f}" for a in adapters))
else:
    print("Chưa có bảng chéo — chạy training/eval_nllb_lora.py")

## Âm Hán Việt cho tên riêng và xưng hô (Trung → Việt, 21/11/2026)

Phim cổ trang đầy tên riêng và xưng hô đọc theo âm Hán Việt (郎君 → lang quân, 霍 → Hoắc, 春风楼 →
Xuân Phong Lâu). Google và NLLB hay để nguyên pinyin ("Lang Jun", "Huo") hoặc dịch sát chữ. Có hai lớp sửa:

1. **Trong mã nguồn**, chạy với mọi engine (`backend/core/hanviet.py`): từ điển âm Hán Việt 13.645 chữ;
   bộ thuật ngữ tự học từ chỗ người dùng sửa tay, từ bộ thuật ngữ đã gõ và từ tên riêng lặp lại trong
   bản dịch AI — cài vào câu gốc trước khi dịch; sau khi dịch, đổi pinyin sót sang âm Hán Việt của đúng
   các chữ trong câu gốc.
2. **Trong adapter Trung** (bản 21/11, đang dùng): huấn luyện lại với thêm 60.000 cặp phụ đề có tên riêng /
   xưng hô Hán Việt, một nửa đã cài sẵn cụm vào câu gốc — mô hình học giữ nguyên cụm đã cài.

| Bộ gold | Adapter bản 11/11 | Adapter bản Hán Việt |
|---|---|---|
| 271 câu, không token: BLEU / chrF | 5,6 / 22,3 | 6,7 / 23,1 |
| 271 câu, bộ siết của app: đơn vị đọc vượt khung | 0/68 | 0/68 |
| Nửa sau (50 cụm), cài cụm đã học + sửa phiên âm: cụm Hán Việt đúng | 6/50 | 9/50 |

BLEU bộ gold +1,07 so với bản 11/11, khoảng 95 % [+0,16; +2,05] (bootstrap 1.000 lượt). Cùng cấu hình,
Google được 12/50 cụm; bộ thuật ngữ người dùng tự gõ cho cả video được 24/50 — cụm Hán
Việt chủ yếu vẫn nhờ bộ thuật ngữ. Chi tiết: `training/README.md`, `training/eval_hanviet.py`.

In [ ]:
from backend.core import hanviet

# âm Hán Việt từng chữ, theo đúng cách đọc của chữ đa âm trong từ điển
print(hanviet.reading("春风楼"))
# bản dịch để sót pinyin → đổi sang âm Hán Việt của đúng các chữ trong câu gốc (không cần mạng hay GPU)
for zh, vi in [("郎君，霍公子来了", "Lang Jun, Huo công tử đến rồi"),
               ("奴家在春风楼等你", "Nô gia đợi anh ở tháp Chunfeng")]:
    print(hanviet.fix_pinyin(zh, vi))